# TakeMeter — Fine-Tuning Starter Notebook
### AI201 · Project 3

This notebook walks you through fine-tuning a text classifier on your annotated dataset and comparing it to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Runs the fine-tuning pipeline with DistilBERT
- Computes evaluation metrics and generates a confusion matrix
- Runs the Groq baseline and compares both models

**What you do (the actual work):**
- Collect and annotate your 200+ examples (done before opening this notebook)
- Define your label map and upload your CSV
- Write your Groq classification prompt using your label definitions
- Analyze the output and write your evaluation report

---
**Before you start:** Make sure you are using a T4 GPU runtime.  
Go to **Runtime → Change runtime type → T4 GPU**, then click Save.

In [ ]:
!pip install -q groq python-dotenv
print("✅ Dependencies ready")


In [ ]:
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


---
## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map.  
Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).

In [ ]:
# TakeMeter label taxonomy — r/QuantumComputing
# news:       linked content is the value (announcement, paper, industry event)
# discussion: poster has a claim/take they want the community to debate
# question:   poster lacks understanding; one person can answer by citing a source

LABEL_MAP = {
    "news":       0,
    "discussion": 1,
    "question":   2,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
print(f"Labels: {LABEL_MAP}")
print(f"Number of labels: {NUM_LABELS}")

In [ ]:
# Upload dataset_v2.csv from your local machine
from google.colab import files
print("Select dataset_v2.csv...")
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {CSV_PATH}")

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"Columns: {df.columns.tolist()}")
print(f"Total examples: {len(df)}")
print()
print("Label distribution:")
print(df["label"].value_counts())

unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    print(f"\n⚠️  Labels in CSV not found in LABEL_MAP: {unknown}")
else:
    print("\n✅ All labels match LABEL_MAP")

df["label_id"] = df["label"].map(LABEL_MAP)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

---
## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.

In [ ]:
# 70 / 15 / 15 stratified split
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"]
)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print()
print("Train label distribution:")
print(train_df["label"].value_counts())
print()
print("Test label distribution:")
print(test_df["label"].value_counts())

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [ ]:
# Load tokenizer and tokenize all splits
# DistilBERT keeps the starter-template architecture. The larger sequence window
# helps with Reddit posts where the title alone is ambiguous.
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 384

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

def make_dataset(df_split):
    ds = Dataset.from_pandas(
        df_split[["text", "label_id"]].rename(columns={"label_id": "labels"})
    )
    return ds.map(tokenize, batched=True)

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization complete")
print(f"Model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_LENGTH}")


---
## Section 3: Fine-Tune Your Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on your training data.  
Training runs for 3 epochs and takes **5–15 minutes** on a T4 GPU.

> **Hyperparameter note:** The defaults below work well for datasets of 100–500 examples.  
> If you change any values, note what you changed and why in your README.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"✅ Model loaded: {MODEL_NAME}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

In [ ]:
# Tuned DistilBERT starter pass:
# - Uses the original HuggingFace Trainer flow and no custom loss.
# - Selects the best checkpoint by validation macro F1 because the project target
#   cares about balanced class quality, not only overall accuracy.
# - Uses a longer input window because title-only intent is often ambiguous.

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.02
GRADIENT_ACCUMULATION_STEPS = 1

steps_per_epoch = int(np.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)))
warmup_steps = max(10, int(0.08 * steps_per_epoch * NUM_EPOCHS))

print("Tuned DistilBERT settings:")
print("  epochs:", NUM_EPOCHS)
print("  learning_rate:", LEARNING_RATE)
print("  train_batch_size:", TRAIN_BATCH_SIZE)
print("  max_length:", MAX_LENGTH)
print("  warmup_steps:", warmup_steps)
print("  weight_decay:", WEIGHT_DECAY)

training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting tuned DistilBERT fine-tuning...")
trainer.train()
print("\nFine-tuning complete")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro F1:", trainer.state.best_metric)


---
## Section 4: Evaluate Fine-Tuned Model

In [ ]:
print("Running inference on test set...")
ft_output = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
ft_macro_f1 = f1_score(ft_true_ids, ft_pred_ids, average="macro", zero_division=0)

print(f"\n🎯 Fine-tuned — Accuracy: {ft_accuracy:.3f}  |  Macro F1: {ft_macro_f1:.3f}")

label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nPer-class metrics (fine-tuned model):")
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))

In [ ]:
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Fine-Tuned {MODEL_NAME} — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("✅ Saved: confusion_matrix.png")


In [ ]:
# Error analysis — review wrong predictions for your README write-up
wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"Wrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for i, idx in enumerate(wrong_idx[:15]):
    text = test_df.iloc[idx]["text"]
    true_label = ID_TO_LABEL[ft_true_ids[idx]]
    pred_label = ID_TO_LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{i+1} ---")
    print(f"Text:      {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True:      {true_label}")
    print(f"Predicted: {pred_label}  (confidence: {confidence:.2f})")
    print()

---
## Section 5: Zero-Shot Baseline (Groq)

Uses `llama-3.3-70b-versatile` with a prompt built from the TakeMeter taxonomy.

In [ ]:
from groq import Groq

# Recommended: Colab Secrets panel (🔑 icon, left sidebar)
# Add secret named GROQ_API_KEY and enable notebook access.
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

assert GROQ_API_KEY, (
    "GROQ_API_KEY not set — add it in the Colab Secrets panel (🔑, left sidebar) "
    "and enable notebook access."
)

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized")

In [ ]:
SYSTEM_PROMPT = """You are classifying posts from the r/QuantumComputing subreddit.
Assign each post to exactly one of the following labels.

news: The post shares an external announcement, paper, or industry development.
The value comes from the linked content, not the poster's own argument or opinion.
The poster may add a brief reaction, but no original reasoning is built.
Example: "Microsoft's Majorana 2 Topological Quantum Computer"
Example: "QpiAI Achieves High-Speed Quantum Error Correction on Superconducting Systems"

discussion: The poster has a perspective, observation, or claim they want the community
to debate. No single correct answer exists — the post is valuable because it invites
expert opinions or analysis.
Example: "When will SC qubits start to die off?"
Example: "What do you think of QuEra's 'Fault-tolerance in 2028' — is it a bold claim?"

question: The poster lacks understanding of something and wants an explanation.
A single well-informed person could answer it correctly by citing a source, paper,
or established concept.
Example: "I don't get generalized amplitude damping"
Example: "Why don't we just perform another transform in the Fourier basis after QFT?"

Respond with ONLY the label name — nothing else.
Valid responses: news, discussion, question"""

print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

In [ ]:
def classify_with_groq(text):
    """Returns a label string or None if the response is unparseable."""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this post:\n\n{text}"},
            ],
            temperature=0,
            max_tokens=10,
        )
        raw = response.choices[0].message.content.strip().lower()
        for label in sorted(LABEL_MAP, key=len, reverse=True):
            if raw == label or label in raw:
                return label
        return None
    except Exception as e:
        print(f"API error: {e}")
        return None


print(f"Running baseline on {len(test_df)} examples...")
print("(~0.1s delay between requests for free-tier rate limits)\n")

baseline_preds = []
for i, (_, row) in enumerate(test_df.iterrows()):
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_df)} complete...")
    time.sleep(0.1)

none_count = baseline_preds.count(None)
if none_count > 0:
    print(f"\n⚠️  {none_count} responses could not be parsed — check your prompt.")

In [ ]:
valid = [(p, t) for p, t in zip(baseline_preds, test_df["label_id"]) if p is not None]
bl_pred_ids = [LABEL_MAP[p] for p, _ in valid]
bl_true_ids = [t for _, t in valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
bl_macro_f1 = f1_score(bl_true_ids, bl_pred_ids, average="macro", zero_division=0)

print(f"🎯 Baseline — Accuracy: {bl_accuracy:.3f}  |  Macro F1: {bl_macro_f1:.3f}")
print(f"   (evaluated on {len(valid)}/{len(test_df)} parseable responses)")
print()
print("Per-class metrics (baseline):")
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))

---
## Section 6: Compare and Export

In [ ]:
print("=" * 55)
print("RESULTS COMPARISON")
print("=" * 55)
print(f"{'Model':<35} {'Accuracy':>8}  {'Macro F1':>8}")
print("-" * 55)
print(f"{'Zero-shot baseline (Groq)':<35} {bl_accuracy:>8.3f}  {bl_macro_f1:>8.3f}")
print(f"{'Fine-tuned DistilBERT':<35} {ft_accuracy:>8.3f}  {ft_macro_f1:>8.3f}")
print("-" * 55)
delta_f1 = ft_macro_f1 - bl_macro_f1
direction = "improvement" if delta_f1 >= 0 else "regression"
print(f"\nMacro F1 {direction}: {abs(delta_f1):.3f}")
print("\nUse these numbers in your README evaluation report.")

In [ ]:
results = {
    "model": MODEL_NAME,
    "label_map": LABEL_MAP,
    "test_set_size": len(test_df),
    "finetuned": {
        "accuracy": round(ft_accuracy, 4),
        "macro_f1": round(ft_macro_f1, 4),
    },
    "baseline": {
        "model": "llama-3.3-70b-versatile",
        "accuracy": round(bl_accuracy, 4),
        "macro_f1": round(bl_macro_f1, 4),
        "parseable": len(valid),
    },
    "improvement": {
        "accuracy": round(ft_accuracy - bl_accuracy, 4),
        "macro_f1": round(ft_macro_f1 - bl_macro_f1, 4),
    },
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Files ready to download:")
print("   evaluation_results.json  — commit to repo, reference in README")
print("   confusion_matrix.png     — embed in README")
print()
print("Download via: Files panel (📁, left sidebar) → right-click → Download")